In [ ]:



import sqlite3
import pandas as pd

raw_data_customers= r"C:\Workspace-Airflow\Mini_Project_1\input_data\customers.csv"
raw_data_orders= r"C:\Workspace-Airflow\Mini_Project_1\input_data\orders.csv"
raw_data_products= r"C:\Workspace-Airflow\Mini_Project_1\input_data\products.csv"
raw_data_clickstream= r"C:\Workspace-Airflow\Mini_Project_1\input_data\clickstream.csv"
df_customers= pd.read_csv(raw_data_customers)
df_orders= pd.read_csv(raw_data_orders)
df_products= pd.read_csv(raw_data_products)
df_clickstream= pd.read_csv(raw_data_clickstream)
conn= sqlite3.connect('database.db')
df_customers.to_sql('customers', conn, if_exists='replace', index=False)
df_orders.to_sql('orders', conn, if_exists='replace', index=False)
df_products.to_sql('products', conn, if_exists='replace', index=False)
df_clickstream.to_sql('clickstream', conn, if_exists='replace', index=False)

pd.read_sql('SELECT * FROM orders', conn)

#top 10 customers by revenue.
pd.read_sql('SELECT o.customer_id, SUM(o.quantity * p.selling_price) AS revenue ' 
'FROM orders o ' 
'JOIN products p ON o.product_id = p.product_id ' 
'GROUP BY o.customer_id ' 
'ORDER BY revenue DESC ' 
'LIMIT 10',conn)

#month-over-month sales growth

query = '''WITH monthly AS (
    SELECT 
        strftime('%Y-%m', o.order_date) AS month,
        SUM(o.quantity * p.selling_price) AS sales
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY month
)
SELECT 
    m1.month,
    m1.sales,
    m2.sales AS prev_month_sales,
    ROUND(((m1.sales - m2.sales) * 100.0) / m2.sales, 2) AS mom_growth_percent
FROM monthly m1
LEFT JOIN monthly m2
    ON m2.month = strftime('%Y-%m', date(m1.month || '-01', '-1 month'))
ORDER BY m1.month;

''' 

pd.read_sql(query, conn)

#customers who ordered in consecutive months.

query1 = '''WITH monthly_orders AS (
    SELECT 
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS month
    FROM orders o
    GROUP BY o.customer_id, month
)
SELECT DISTINCT
    m1.customer_id
FROM monthly_orders m1
JOIN monthly_orders m2
    ON m1.customer_id = m2.customer_id
    AND m2.month = strftime('%Y-%m', date(m1.month || '-01', '-1 month'))
ORDER BY m1.customer_id;
'''

pd.read_sql(query1, conn)

#products never ordered.

query2 = '''SELECT p.product_id, p.product_name
FROM products p
LEFT JOIN orders o ON p.product_id = o.product_id
WHERE o.product_id IS NULL
'''
pd.read_sql(query2, conn)

#revenue contribution percentage by category

query3 = '''WITH category_revenue AS (
    SELECT 
        p.category,
        SUM(o.quantity * p.selling_price) AS revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category
),
total AS (
    SELECT SUM(revenue) AS total_revenue
    FROM category_revenue
)
SELECT 
    cr.category,
    cr.revenue,
    ROUND((cr.revenue * 100.0) / t.total_revenue, 2) AS revenue_percentage
FROM category_revenue cr, total t
ORDER BY revenue_percentage DESC
'''
pd.read_sql(query3, conn)

#Rank customers based on total revenue

query4 = ''' SELECT o.customer_id, SUM(o.quantity * p.selling_price) AS total_revenue
FROM orders o
JOIN products p ON o.product_id =p.product_id
GROUP BY o.customer_id
ORDER BY total_revenue DESC'''


pd.read_sql(query4, conn)

# running total sales by month

query5 = '''WITH monthly_sales AS (
    SELECT 
        strftime('%Y-%m', o.order_date) AS month,
        SUM(o.quantity * p.selling_price) AS sales
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY month 
)
SELECT 
    month,
    sales,
    SUM(sales) OVER (ORDER BY month) AS running_total
FROM monthly_sales
ORDER BY month;
'''
pd.read_sql(query5, conn)

#highest selling product per category

query6 = '''WITH category_sales AS (
    SELECT 
        p.category,
        p.product_id,
        p.product_name,
        SUM(o.quantity) AS total_quantity_sold
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category, p.product_id, p.product_name
)
SELECT 
    cs.category,
    cs.product_id,
    cs.product_name,
    cs.total_quantity_sold
FROM category_sales cs
JOIN (
    SELECT 
        category,
        MAX(total_quantity_sold) AS max_quantity
    FROM category_sales
    GROUP BY category
) max_sales 
ON cs.category = max_sales.category 
AND cs.total_quantity_sold = max_sales.max_quantity
ORDER BY cs.category;
'''
pd.read_sql(query6, conn)

#7-day rolling average sales.

query7 = '''WITH daily_sales AS (
    SELECT 
        date(o.order_date) AS order_day,
        SUM(o.quantity * p.selling_price) AS sales
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY order_day
)
SELECT 
    d1.order_day,
    d1.sales,
    (
        SELECT ROUND(AVG(d2.sales), 2)
        FROM daily_sales d2
        WHERE d2.order_day BETWEEN date(d1.order_day, '-6 day') AND d1.order_day
    ) AS rolling_7_day_avg
FROM daily_sales d1
ORDER BY d1.order_day;

'''
pd.read_sql(query7, conn)




,order_day,sales,rolling_7_day_avg
0,2024-09-26,698.0,698.0
1,2024-09-30,9996.0,5347.0
2,2024-10-06,9995.0,9995.5
3,2024-10-15,799.0,799.0
4,2024-11-05,3998.0,3998.0
...,...,...,...
395,2028-02-22,1598.0,1598.0
396,2028-03-03,998.0,998.0
397,2028-03-26,2598.0,2598.0
398,2028-03-29,1999.0,2298.5
